# CSCI 3151 – Foundations of Machine Learning

## M49 – LSTMs, GRUs, and gated RNN variants (mini-lab)

In this mini-lab you will work with a simple synthetic sequence dataset: 1D random walks of fixed length. Each sequence starts from an initial *seed* value, then wanders via Gaussian noise. Your job is to train small recurrent models that see the **entire sequence** and try to reconstruct the original seed value from the final hidden state.

This is a controlled **long-range dependency** task: the only information that really matters is at time step 0, but the model only makes a prediction after many steps.

**This mini-lab reinforces:**

- Why vanilla RNNs struggle to carry information across many time steps (vanishing gradients).
- How LSTMs and GRUs use gates to remember, forget, and expose information.
- How to implement and train gated RNNs in PyTorch on a long-range regression task.
- How to compare models empirically using validation loss and simple baselines.


In [ ]:
# Setup: imports, device, and reproducibility

import numpy as np
import torch
from torch import nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

SEED = 3151

np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


In [ ]:
# Generate random-walk data for a long-range memory task

def make_walk_data(n, T, sigma, rng):
    # Generate n random-walk sequences of length T.
    # Each sequence starts at x0 ~ Uniform(-1, 1) and then
    # evolves as x_t = x_{t-1} + noise_t with Gaussian noise.
    # The regression target is the initial value x0.
    x0 = rng.uniform(-1.0, 1.0, size=(n,)).astype("float32")
    noise = rng.normal(0.0, sigma, size=(n, T)).astype("float32")
    seqs = np.zeros((n, T), dtype="float32")
    seqs[:, 0] = x0
    for t in range(1, T):
        seqs[:, t] = seqs[:, t - 1] + noise[:, t]
    # Add a final feature dimension so each time step has shape (1,)
    return seqs[:, :, np.newaxis], x0

rng = np.random.default_rng(SEED)

# Feel free to shrink these if training is slow on your laptop.
N_TRAIN = 2000
N_VAL = 500
T_SEQ = 100  # sequence length (number of time steps)
SIGMA = 0.1  # noise scale

X_train_np, y_train_np = make_walk_data(N_TRAIN, T_SEQ, SIGMA, rng)
X_val_np, y_val_np = make_walk_data(N_VAL, T_SEQ, SIGMA, rng)

print("X_train shape:", X_train_np.shape)
print("y_train shape:", y_train_np.shape)

# Convert to PyTorch tensors and build DataLoaders
X_train = torch.from_numpy(X_train_np)
y_train = torch.from_numpy(y_train_np).unsqueeze(1)  # (N, 1)
X_val = torch.from_numpy(X_val_np)
y_val = torch.from_numpy(y_val_np).unsqueeze(1)

BATCH_SIZE = 64

train_ds = TensorDataset(X_train, y_train)
val_ds = TensorDataset(X_val, y_val)
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE)


In [ ]:
# Quick look at a few random-walk sequences

fig, ax = plt.subplots()
for i in range(5):
    ax.plot(X_train_np[i, :, 0], alpha=0.7)
ax.set_xlabel("time step")
ax.set_ylabel("value")
ax.set_title("Random-walk sequences (target = value at step 0)")
plt.tight_layout()


## Core task 1 — Understand the dataset

Using the plot above (and `X_train_np` / `y_train_np` if you like):

- Identify where in the sequence the **useful information** for the prediction lives.
- Describe in words why this is a **long-range dependency** problem.
- Explain why a model that only looks at the **final observation** would generally do poorly at recovering the initial value.

If you have time, jot down a few bullet points in a new Markdown cell with your answers.


In [ ]:
# Simple numerical baselines for reference

# Baseline 1: always predict 0 (ignores the sequence entirely)
naive_pred = np.zeros_like(y_val_np)
naive_mse = float(np.mean((y_val_np - naive_pred) ** 2))

# Baseline 2: predict the last value in the random walk
last_step_pred = X_val_np[:, -1, 0]
last_step_mse = float(np.mean((y_val_np - last_step_pred) ** 2))

print(f"Naive 'always 0' baseline val MSE: {naive_mse:.4f}")
print(f"Using only last step x_T baseline val MSE: {last_step_mse:.4f}")

print("We'll want our recurrent models to beat at least the naive baseline.")


## Core task 2 — Train an LSTM regressor

We will now build a small PyTorch model that reads the entire sequence and predicts the seed value.

The high-level design:

1. Feed the sequence (shape `(batch, T, 1)`) into a recurrent layer.
2. Take the **final hidden state** from the last time step.
3. Pass it through a linear layer to produce a single scalar prediction.

In the code cells below you will:

- Define a reusable `RandomWalkRegressor` class that can use a vanilla RNN, GRU, or LSTM cell.
- Train an **LSTM** version on the random-walk task.
- Compare its validation MSE against the naive baselines from the previous cell.

Try to focus on **what the model is doing conceptually** (carrying information across time) rather than getting bogged down in PyTorch details.


In [ ]:
# Model class: vanilla RNN / GRU / LSTM on top of the random walk

class RandomWalkRegressor(nn.Module):
    def __init__(self, cell_type="lstm", input_dim=1, hidden_dim=32):
        super().__init__()
        self.cell_type = cell_type.lower()
        if self.cell_type not in {"rnn", "gru", "lstm"}:
            raise ValueError("cell_type must be 'rnn', 'gru', or 'lstm'")

        if self.cell_type == "rnn":
            self.rnn = nn.RNN(input_dim, hidden_dim, batch_first=True)
        elif self.cell_type == "gru":
            self.rnn = nn.GRU(input_dim, hidden_dim, batch_first=True)
        else:  # LSTM
            self.rnn = nn.LSTM(input_dim, hidden_dim, batch_first=True)

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        # x has shape (batch, time, input_dim)
        output, hidden = self.rnn(x)
        # For LSTM, hidden is a (h_n, c_n) tuple; for RNN/GRU it is just h_n
        if self.cell_type == "lstm":
            h_n, c_n = hidden
        else:
            h_n = hidden
        # h_n has shape (num_layers, batch, hidden_dim); take final layer
        last_hidden = h_n[-1]
        out = self.fc(last_hidden)
        return out

def count_params(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
# Training loop for the regression task

def train_regressor(model, train_loader, val_loader, epochs=10, lr=1e-3):
    model.to(device)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()

    history = {"epoch": [], "train_loss": [], "val_loss": []}

    for ep in range(1, epochs + 1):
        model.train()
        train_loss_sum = 0.0
        n_train = 0
        for xb, yb in train_loader:
            xb = xb.to(device)
            yb = yb.to(device)

            opt.zero_grad()
            preds = model(xb)
            loss = loss_fn(preds, yb)
            loss.backward()
            # Gradient clipping helps stabilise RNN training
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
            opt.step()

            train_loss_sum += float(loss.item()) * xb.size(0)
            n_train += xb.size(0)

        train_loss = train_loss_sum / max(1, n_train)

        model.eval()
        val_loss_sum = 0.0
        n_val = 0
        with torch.no_grad():
            for xb, yb in val_loader:
                xb = xb.to(device)
                yb = yb.to(device)
                preds = model(xb)
                loss = loss_fn(preds, yb)
                val_loss_sum += float(loss.item()) * xb.size(0)
                n_val += xb.size(0)

        val_loss = val_loss_sum / max(1, n_val)

        history["epoch"].append(ep)
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)

        print(f"Epoch {ep:02d}: train MSE = {train_loss:.4f}, val MSE = {val_loss:.4f}")

    return history


In [ ]:
# Train an LSTM on the random-walk task

HIDDEN_DIM = 32
EPOCHS = 10  # If this is slow, try reducing to 5

lstm_model = RandomWalkRegressor(cell_type="lstm", hidden_dim=HIDDEN_DIM)
print(f"LSTM parameters: {count_params(lstm_model):,}")

lstm_history = train_regressor(lstm_model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3)

# Plot training vs validation loss and compare to the naive baseline
plt.figure()
plt.plot(lstm_history["epoch"], lstm_history["train_loss"], label="LSTM train")
plt.plot(lstm_history["epoch"], lstm_history["val_loss"], label="LSTM val")
plt.axhline(naive_mse, linestyle="--", label="naive always-0 baseline")
plt.xlabel("epoch")
plt.ylabel("MSE on x0")
plt.legend()
plt.tight_layout()


## Core task 3 — Compare LSTM to a vanilla RNN (and optionally a GRU)

Now that you have an LSTM trained:

1. Instantiate a **vanilla RNN** model by setting `cell_type="rnn"`.
2. Train it with the same `HIDDEN_DIM`, `EPOCHS`, and learning rate.
3. Plot its validation loss curve on top of the LSTM curve and compare both to the naive baseline.

As you compare, ask yourself:

- Does the vanilla RNN beat the naive baseline?
- How close does it get to the LSTM's validation loss?
- Do you see evidence that the LSTM is better at carrying information about `x0` across 100 steps?

If you have time, repeat the same experiment with `cell_type="gru"` and see whether the GRU behaves more like the RNN or the LSTM.


In [ ]:
# TODO: train a vanilla RNN (and optionally a GRU) on the same task

rnn_model = RandomWalkRegressor(cell_type="rnn", hidden_dim=HIDDEN_DIM)
print(f"RNN parameters: {count_params(rnn_model):,}")

# Uncomment the line below to train the RNN.
# rnn_history = train_regressor(rnn_model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3)

# After training, you can compare validation losses, for example:
# plt.figure()
# plt.plot(lstm_history["epoch"], lstm_history["val_loss"], label="LSTM val")
# plt.plot(rnn_history["epoch"], rnn_history["val_loss"], label="RNN val")
# plt.axhline(naive_mse, linestyle="--", label="naive always-0 baseline")
# plt.xlabel("epoch")
# plt.ylabel("MSE on x0")
# plt.legend()
# plt.tight_layout()

# OPTIONAL: try a GRU as well by changing cell_type to "gru".
# gru_model = RandomWalkRegressor(cell_type="gru", hidden_dim=HIDDEN_DIM)
# gru_history = train_regressor(gru_model, train_loader, val_loader, epochs=EPOCHS, lr=1e-3)


## Stretch tasks (if you finish early)

If you get through the core tasks with time to spare, try one or more of these:

1. **Vary the sequence length.** Regenerate the data with a shorter sequence (e.g., `T_SEQ = 30`) and with a longer one (e.g., `T_SEQ = 150`). Re-train the RNN and LSTM and see how their relative performance changes as the long-range dependency gets harder.
2. **Parameter counts.** Use `count_params` to build a tiny table with parameter counts for the RNN, GRU, and LSTM (for the same `hidden_dim`). Relate this to the rule of thumb from the video about when extra gates are worth the cost.
3. **Hyperparameters.** Try changing `hidden_dim`, learning rate, or batch size. Do any of these make the vanilla RNN behave more like the LSTM, or does the gating structure still matter most for very long sequences?

As you explore, keep linking your observations back to the **gate roles** from the M49 deck: forget gates preserving long-term memory, input gates writing new information, and output/update gates deciding what to expose.
